# Training Mask R-CNN (torchvision) on SUN RGB-D

The two-stage `maskrcnn_resnet50_fpn_v2` model from torchvision, transfer learning from weights pre-trained on COCO.

Class configs:

| `CONFIG` | classes | description |
|---|---|---|
| `std10`  | 10 | standard SUN RGB-D benchmark (comparable with publications) |
| `min400` | 20 | std10 + every class with ≥ 400 instances after merges/exclusions |
| `min100` | 41 | std10 + every class with ≥ 100 instances after merges/exclusions |

In [ ]:
from pathlib import Path
import math, time, json
import random
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
from pycocotools.cocoeval import COCOeval

from torchvision import tv_tensors
from torchvision.transforms import v2
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

from contextlib import nullcontext

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# class config to train; the archive holds the splits of all three
CONFIG = "std10"          # "std10" (10 classes) | "min400" (20) | "min100" (41)
SPLIT_SUBDIR = {"std10": "splits", "min400": "splits_min400", "min100": "splits_min100"}[CONFIG]

DATA_ZIP  = Path("/content/drive/MyDrive/sunrgbd_data.zip")    # archive path on Drive
DATA_ROOT = Path("/content/sunrgbd")                           # where to unpack
OUT_DIR   = Path(f"/content/drive/MyDrive/vkr_runs/maskrcnn_torchvision_{CONFIG}")
SPLIT_DIR = DATA_ROOT / SPLIT_SUBDIR

# unpack once per Colab session (the zip is stored uncompressed, ~5 min from Drive)
if not SPLIT_DIR.is_dir():
    assert DATA_ZIP.is_file(), f"archive not found on Drive: {DATA_ZIP}"
    print("Unpacking archive...")
    !unzip -q -n "{DATA_ZIP}" -d "{DATA_ROOT}"

OUT_DIR.mkdir(parents=True, exist_ok=True)
for s in ("train", "val", "test"):
    assert (SPLIT_DIR / f"instances_{s}.json").is_file(), f"missing instances_{s}.json"
print("Config:", CONFIG)
print("Data:", DATA_ROOT)
print("Annotations:", SPLIT_DIR)
print("Outputs:", OUT_DIR)

In [ ]:
SEED          = 42
N_THING_CLASSES = len(json.loads(
    (SPLIT_DIR / "instances_train.json").read_text())["categories"])
NUM_CLASSES   = N_THING_CLASSES + 1   # thing classes + background
BATCH_SIZE    = 4             
NUM_WORKERS   = 2
EPOCHS        = 12
BASE_LR       = 0.005         # SGD; linearly scaled from 0.02 at batch 16
MOMENTUM      = 0.9
WEIGHT_DECAY  = 5e-4
LR_MILESTONES = [8, 11]       # epochs at which LR is divided by 10
WARMUP_ITERS  = 500
SCORE_THRESH_VIS = 0.5        # confidence threshold for visualizations only

In [ ]:
random.seed(SEED) 
np.random.seed(SEED) 
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{N_THING_CLASSES} thing classes + background = NUM_CLASSES {NUM_CLASSES}")

---
## Dataset

A wrapper class over the COCO files prepared. Masks are stored as
polygons and rasterized on the fly via `pycocotools`.

In [ ]:
class SunRgbdDataset(torch.utils.data.Dataset):
    """COCO-style SUN RGB-D dataset for Mask R-CNN."""

    def __init__(self, ann_file, img_root, transforms=None):
        self.coco = COCO(str(ann_file))
        self.img_root = Path(img_root)
        self.ids = sorted(self.coco.getImgIds())
        self.transforms = transforms

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id, iscrowd=False))

        img = Image.open(self.img_root / info["file_name"]).convert("RGB")
        w, h = img.size

        boxes, labels, masks = [], [], []
        for a in anns:
            x, y, bw, bh = a["bbox"]
            if bw < 1 or bh < 1:
                continue
            boxes.append([x, y, x + bw, y + bh])
            labels.append(a["category_id"])
            masks.append(self.coco.annToMask(a))

        img = tv_tensors.Image(img)
        target = {
            "boxes": tv_tensors.BoundingBoxes(
                torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                format="XYXY", canvas_size=(h, w)),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": tv_tensors.Mask(
                torch.as_tensor(np.stack(masks), dtype=torch.uint8)),
            "image_id": img_id,
        }
        if self.transforms is not None:
            img, target = self.transforms(img, target)
        return img, target


train_tf = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToDtype(torch.float32, scale=True),
])
eval_tf = v2.Compose([v2.ToDtype(torch.float32, scale=True)])


def collate(batch):
    return tuple(zip(*batch))


ds_train = SunRgbdDataset(SPLIT_DIR / "instances_train.json", DATA_ROOT, train_tf)
ds_val   = SunRgbdDataset(SPLIT_DIR / "instances_val.json",   DATA_ROOT, eval_tf)
ds_test  = SunRgbdDataset(SPLIT_DIR / "instances_test.json",  DATA_ROOT, eval_tf)

dl_train = torch.utils.data.DataLoader(
    ds_train, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0)
dl_val = torch.utils.data.DataLoader(
    ds_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
dl_test = torch.utils.data.DataLoader(
    ds_test, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

CLASS_NAMES = {c["id"]: c["name"] for c in ds_train.coco.loadCats(ds_train.coco.getCatIds())}
print(f"train {len(ds_train)}  val {len(ds_val)}  test {len(ds_test)}")
print("Classes:", CLASS_NAMES)

In [ ]:
# visual check: images after augmentations + masks

PALETTE = np.array([
    [230,25,75],[60,180,75],[255,225,25],[0,130,200],[245,130,48],
    [145,30,180],[70,240,240],[240,50,230],[210,245,60],[0,128,128],
], dtype=np.float32) / 255.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axes, [0, 1, 2]):
    img, tgt = ds_train[k]
    im = img.permute(1, 2, 0).numpy().copy()
    for i, m in enumerate(tgt["masks"].numpy()):
        im[m.astype(bool)] = 0.5 * im[m.astype(bool)] + 0.5 * PALETTE[i % len(PALETTE)]
    ax.imshow(im)
    ax.set_title(", ".join(sorted({CLASS_NAMES[int(l)] for l in tgt["labels"]}))[:60], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## Model

`maskrcnn_resnet50_fpn_v2` — the improved Mask R-CNN implementation: ResNet-50 + FPN backbone, weights pre-trained on COCO.

For transfer learning two heads are replaced:
* `box_predictor` — box classification and regression (11 classes: 10 + background);
* `mask_predictor` — mask prediction.

The remaining weights are initialized from COCO and fine-tuned.

In [ ]:
def build_model(num_classes: int):
    model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)

    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)

    in_feat_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_feat_mask, 256, num_classes)
    return model


model = build_model(NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Parameters: {n_params:.1f} M")

---
## Training

* **Optimizer:** SGD with momentum 0.9 (the standard for the R-CNN family).
* **Warmup:** for the first 500 iterations the LR grows linearly from 1/1000 of
  the base value — stabilizes the early steps while the new heads are still random.
* **LR schedule:** step decay, /10 at epochs 8 and 11.
* **Mixed precision (AMP)** — speeds up training on a T4 roughly twofold.
* **Checkpoints:** `last.pth` (to resume after a session drop) and `best.pth`
  (highest validation mask AP) are stored on Google Drive.
* **Model selection metric:** COCO **mask AP** (mean over IoU 0.50:0.95) on the
  validation set. The test set is not touched until the final evaluation.

In [ ]:
AMP = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=AMP)


def train_one_epoch(model, optimizer, loader, epoch, warmup_iters=0):
    model.train()
    warmup = None
    if epoch == 0 and warmup_iters > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=1e-3, total_iters=warmup_iters)

    running, t0 = {}, time.time()
    for it, (images, targets) in enumerate(loader):
        images = [im.to(DEVICE, non_blocking=True) for im in images]
        targets = [{k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in t.items()}
                   for t in targets]

        with torch.amp.autocast("cuda") if AMP else nullcontext():
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        if not math.isfinite(loss.item()):
            print("Non-finite loss, batch skipped:", {k: v.item() for k, v in loss_dict.items()})
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if warmup is not None:
            warmup.step()

        for k, v in loss_dict.items():
            running[k] = running.get(k, 0.0) + v.item()
        running["loss"] = running.get("loss", 0.0) + loss.item()

        if (it + 1) % 100 == 0:
            speed = (it + 1) / (time.time() - t0)
            print(f"  ep.{epoch+1} it.{it+1}/{len(loader)} "
                  f"loss={running['loss']/(it+1):.3f} "
                  f"({speed:.2f} it/s, LR={optimizer.param_groups[0]['lr']:.5f})", end="\r")
    print()
    return {k: v / len(loader) for k, v in running.items()}


@torch.no_grad()
def evaluate_coco(model, loader, dataset):
    """Inference + COCOeval. Returns (metrics, per-class AP, raw results)."""
    model.eval()
    results = []
    for images, targets in loader:
        images = [im.to(DEVICE, non_blocking=True) for im in images]
        with torch.amp.autocast("cuda") if AMP else nullcontext():
            outputs = model(images)
        for tgt, out in zip(targets, outputs):
            img_id = int(tgt["image_id"])
            boxes = out["boxes"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            labels = out["labels"].cpu().numpy()
            masks = (out["masks"].cpu().numpy() > 0.5).astype(np.uint8)  # [N,1,H,W]
            for box, score, label, m in zip(boxes, scores, labels, masks):
                rle = mask_utils.encode(np.asfortranarray(m[0]))
                rle["counts"] = rle["counts"].decode("ascii")
                x1, y1, x2, y2 = box
                results.append({
                    "image_id": img_id, "category_id": int(label),
                    "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                    "score": float(score), "segmentation": rle,
                })
    if not results:
        return {"segm_AP": 0.0, "bbox_AP": 0.0}, {}, results

    coco_gt = dataset.coco
    coco_dt = coco_gt.loadRes(results)

    metrics, per_class = {}, {}
    for iou_type in ("segm", "bbox"):
        ev = COCOeval(coco_gt, coco_dt, iou_type)
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics[f"{iou_type}_AP"]   = ev.stats[0]
        metrics[f"{iou_type}_AP50"] = ev.stats[1]
        metrics[f"{iou_type}_AP75"] = ev.stats[2]
        metrics[f"{iou_type}_APs"]  = ev.stats[3]
        metrics[f"{iou_type}_APm"]  = ev.stats[4]
        metrics[f"{iou_type}_APl"]  = ev.stats[5]
        if iou_type == "segm":
            # per-class AP: precision has shape [IoU, recall, class, area, maxDet]
            prec = ev.eval["precision"]
            for k, cat_id in enumerate(ev.params.catIds):
                p = prec[:, :, k, 0, -1]
                per_class[CLASS_NAMES[cat_id]] = float(p[p > -1].mean()) if (p > -1).any() else float("nan")
    return metrics, per_class, results

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=BASE_LR,
                            momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
lr_sched = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=LR_MILESTONES, gamma=0.1)

LAST_CKPT, BEST_CKPT = OUT_DIR / "last.pth", OUT_DIR / "best.pth"
HIST_CSV = OUT_DIR / "history.csv"

# resume after a Colab session drop
start_epoch, best_ap, history = 0, -1.0, []
if LAST_CKPT.is_file():
    ck = torch.load(LAST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    lr_sched.load_state_dict(ck["lr_sched"])
    scaler.load_state_dict(ck["scaler"])
    start_epoch, best_ap, history = ck["epoch"] + 1, ck["best_ap"], ck["history"]
    print(f"Resumed from checkpoint: epoch {start_epoch}, best mask AP={best_ap:.4f}")

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_losses = train_one_epoch(model, optimizer, dl_train, epoch,
                                   warmup_iters=WARMUP_ITERS)
    lr_sched.step()

    val_metrics, val_per_class, _ = evaluate_coco(model, dl_val, ds_val)

    row = {"epoch": epoch + 1, "lr": optimizer.param_groups[0]["lr"],
           "time_min": (time.time() - t0) / 60,
           **{f"train_{k}": v for k, v in train_losses.items()},
           **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    pd.DataFrame(history).to_csv(HIST_CSV, index=False)

    ck = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
          "lr_sched": lr_sched.state_dict(), "scaler": scaler.state_dict(),
          "epoch": epoch, "best_ap": best_ap, "history": history,
          "class_names": CLASS_NAMES}
    torch.save(ck, LAST_CKPT)
    if val_metrics["segm_AP"] > best_ap:
        best_ap = val_metrics["segm_AP"]
        ck["best_ap"] = best_ap
        torch.save(ck, BEST_CKPT)

    print(f"Epoch {epoch+1}/{EPOCHS}: loss={train_losses['loss']:.3f} "
          f"mask AP={val_metrics['segm_AP']:.4f} (best {best_ap:.4f}) "
          f"box AP={val_metrics['bbox_AP']:.4f} [{row['time_min']:.1f} min]")

print("\nTraining finished. Best validation mask AP:", round(best_ap, 4))

In [ ]:
# training curves — material for chapter 2
hist = pd.read_csv(HIST_CSV)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].plot(hist.epoch, hist.train_loss, "-o", ms=3, color="#2f6f9f")
axes[0].set_xlabel("Epoch"); axes[0].set_title("Total loss (train)"); axes[0].grid(alpha=0.3)

loss_parts = [c for c in hist.columns if c.startswith("train_loss_")]
for c in loss_parts:
    axes[1].plot(hist.epoch, hist[c], "-o", ms=2, label=c.replace("train_loss_", ""))
axes[1].set_xlabel("Epoch"); axes[1].set_title("Loss components"); axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)

axes[2].plot(hist.epoch, hist.val_segm_AP, "-o", ms=3, label="mask AP", color="#c0392b")
axes[2].plot(hist.epoch, hist.val_bbox_AP, "-o", ms=3, label="box AP", color="#2f6f9f")
axes[2].plot(hist.epoch, hist.val_segm_AP50, "--", lw=1, label="mask AP50", color="#c0392b")
axes[2].set_xlabel("Epoch"); axes[2].set_title("Validation quality"); axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Final evaluation on the test set

Uses the **best model by validation** (`best.pth`). 

In [ ]:
ck = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"])
print(f"Loaded best.pth (epoch {ck['epoch']+1}, val mask AP={ck['best_ap']:.4f})\n")

test_metrics, test_per_class, test_results = evaluate_coco(model, dl_test, ds_test)

summary = pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}).round(4)
summary.to_csv(OUT_DIR / "test_metrics.csv")
print("\nFinal metrics (test):"); display(summary)

per_class_df = (pd.Series(test_per_class, name="mask AP").sort_values(ascending=False)
                .to_frame().round(4))
per_class_df.to_csv(OUT_DIR / "test_per_class_ap.csv")
print("Per-class mask AP:"); display(per_class_df)

with open(OUT_DIR / "test_predictions.json", "w") as f:
    json.dump(test_results, f)
print("Predictions saved:", OUT_DIR / "test_predictions.json")

In [ ]:
# extra test metrics from the saved predictions (no GPU needed):
# per-class precision / recall / F1, confusion matrix, mean IoU of matched masks
import numpy as np, json
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

CONF_THRESH = 0.5   # confidence threshold (P/R at fixed conf, like YOLO reports)
IOU_THRESH  = 0.5   # mask IoU threshold for matching predictions to GT

coco_gt = COCO(str(SPLIT_DIR / "instances_test.json"))
PRED_FILE = OUT_DIR / "test_predictions.json"
preds = [p for p in json.loads(PRED_FILE.read_text()) if p["score"] >= CONF_THRESH]

cat_ids   = sorted(coco_gt.getCatIds())
cat_names = [coco_gt.loadCats(c)[0]["name"] for c in cat_ids]
cat_idx   = {c: i for i, c in enumerate(cat_ids)}
K = len(cat_ids)

# confusion matrix: rows = GT classes (+ background), cols = predicted (+ missed)
conf = np.zeros((K + 1, K + 1), dtype=int)
tp, fp, fn = np.zeros(K, int), np.zeros(K, int), np.zeros(K, int)
matched_ious = []

preds_by_img = {}
for p in preds:
    preds_by_img.setdefault(p["image_id"], []).append(p)

for img_id in coco_gt.getImgIds():
    gts = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_id, iscrowd=False))
    dts = sorted(preds_by_img.get(img_id, []), key=lambda p: -p["score"])
    gt_rles = [coco_gt.annToRLE(a) for a in gts]
    dt_rles = [d["segmentation"] for d in dts]
    ious = (mask_utils.iou(dt_rles, gt_rles, [0] * len(gt_rles))
            if gt_rles and dt_rles else np.zeros((len(dt_rles), len(gt_rles))))

    # greedy class-agnostic matching by descending score
    gt_used = set()
    for di, d in enumerate(dts):
        best_gi, best_iou = -1, IOU_THRESH
        for gi in range(len(gts)):
            if gi not in gt_used and ious[di, gi] >= best_iou:
                best_gi, best_iou = gi, ious[di, gi]
        dk = cat_idx[d["category_id"]]
        if best_gi >= 0:
            gt_used.add(best_gi)
            gk = cat_idx[gts[best_gi]["category_id"]]
            conf[gk, dk] += 1
            matched_ious.append(best_iou)
            if gk == dk:
                tp[dk] += 1
            else:                     # localized but misclassified
                fp[dk] += 1; fn[gk] += 1
        else:                         # no GT match -> false positive on background
            conf[K, dk] += 1
            fp[dk] += 1
    for gi, g in enumerate(gts):      # unmatched GT -> missed object
        if gi not in gt_used:
            gk = cat_idx[g["category_id"]]
            conf[gk, K] += 1
            fn[gk] += 1

prec = tp / np.maximum(tp + fp, 1)
rec  = tp / np.maximum(tp + fn, 1)
f1   = 2 * prec * rec / np.maximum(prec + rec, 1e-9)

table = pd.DataFrame({"precision": prec, "recall": rec, "F1": f1,
                      "TP": tp, "FP": fp, "FN": fn}, index=cat_names)
mp = tp.sum() / max(tp.sum() + fp.sum(), 1)
mr = tp.sum() / max(tp.sum() + fn.sum(), 1)
table.loc["micro avg"] = [mp, mr, 2 * mp * mr / max(mp + mr, 1e-9),
                          tp.sum(), fp.sum(), fn.sum()]
table.loc["macro avg"] = [prec.mean(), rec.mean(), f1.mean(),
                          tp.sum(), fp.sum(), fn.sum()]
table = table.round(4)
table.to_csv(OUT_DIR / "test_pr_f1.csv")
print(f"conf >= {CONF_THRESH}, mask IoU >= {IOU_THRESH}")
print(f"Mean IoU of matched masks: {np.mean(matched_ious):.4f}")
display(table)

# confusion matrix normalized by GT row
norm = conf / np.maximum(conf.sum(axis=1, keepdims=True), 1)
row_lbl = cat_names + ["background"]
col_lbl = cat_names + ["missed"]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(K + 1)); ax.set_xticklabels(col_lbl, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(K + 1)); ax.set_yticklabels(row_lbl, fontsize=8)
for i in range(K + 1):
    for j in range(K + 1):
        if conf[i, j]:
            ax.text(j, i, conf[i, j], ha="center", va="center", fontsize=7,
                    color="white" if norm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Confusion matrix (mask IoU >= {IOU_THRESH}, conf >= {CONF_THRESH})")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# qualitative examples: predictions vs ground truth (for the thesis text)
import random as _random

rng = _random.Random(SEED)
sample_ids = rng.sample(range(len(ds_test)), 4)

model.eval()
fig, axes = plt.subplots(len(sample_ids), 2, figsize=(12, 5 * len(sample_ids)))
for row, idx in enumerate(sample_ids):
    img, tgt = ds_test[idx]
    with torch.no_grad():
        out = model([img.to(DEVICE)])[0]

    base = img.permute(1, 2, 0).numpy()

    gt = base.copy()
    for i, m in enumerate(tgt["masks"].numpy()):
        gt[m.astype(bool)] = 0.5 * gt[m.astype(bool)] + 0.5 * PALETTE[i % len(PALETTE)]
    axes[row, 0].imshow(gt); axes[row, 0].axis("off")
    axes[row, 0].set_title("Ground truth", fontsize=9)

    pred = base.copy()
    keep = out["scores"] > SCORE_THRESH_VIS
    names = []
    for i, m in enumerate(out["masks"][keep].cpu().numpy()):
        sel = m[0] > 0.5
        pred[sel] = 0.5 * pred[sel] + 0.5 * PALETTE[i % len(PALETTE)]
        names.append(CLASS_NAMES[int(out["labels"][keep][i])])
    axes[row, 1].imshow(pred); axes[row, 1].axis("off")
    axes[row, 1].set_title(f"Mask R-CNN (score>{SCORE_THRESH_VIS}): " +
                           ", ".join(sorted(set(names)))[:60], fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "qualitative_test.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Inference timing

The protocol is identical across notebooks 02/03/04 (section 2.3 compares the
numbers directly): end-to-end single-image latency from a decoded image in
memory — preprocessing + forward pass + postprocessing, batch = 1,
10 warmup runs, 100 timed runs on test images, GPU-synchronized timing.

In [ ]:
N_WARMUP, N_RUNS = 10, 100
model.eval()
sample = [ds_test[i][0] for i in range(min(N_WARMUP + N_RUNS, len(ds_test)))]

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

with torch.no_grad():
    for img in sample[:N_WARMUP]:
        model([img.to(DEVICE)])
    _sync()
    times = []
    for img in sample[N_WARMUP:]:
        _sync(); t0 = time.perf_counter()
        model([img.to(DEVICE)])
        _sync(); times.append(time.perf_counter() - t0)

times = np.array(times) * 1000
timing = {"model": f"maskrcnn_torchvision_{CONFIG}",
          "device": (torch.cuda.get_device_name(0)
                     if torch.cuda.is_available() else "cpu"),
          "n_runs": len(times),
          "mean_ms": float(times.mean()), "std_ms": float(times.std()),
          "median_ms": float(np.median(times)),
          "fps": float(1000.0 / times.mean())}
(OUT_DIR / "timing.json").write_text(json.dumps(timing, indent=2))
print(json.dumps(timing, indent=2))